# 40 — Run Blind-B inference + dev no-regression smoke

Per RecSys_Challenge_Plan §W7 + `project_codabench_submission.md`. Orchestrates the final pipeline against the W7-merged 7B from notebook 33:

1. **Dev no-regression smoke**: re-run inference on the dev split (`talkpl-ai/TalkPlayData-Challenge-Dataset` test split, 8000 rows) with config 220 (which now points at the W7 merged repo). Compares dev nDCG@20 against the W3 retrieval-only baseline. Plan §6.6 hard rule: must NOT regress > 0.005.
2. **Blind-B inference**: run `run_inference_blindset.py --tid 300-final-blindset-B` against the live Blind-B dataset.
3. **Validate + package**: run `scripts/validate_prediction.py --split blindB --package` to produce the CodaBench-ready zip with `prediction.json` at root (per `project_codabench_submission.md`).

## Sequence

1. Mount Drive, install deps, HF auth.
2. Read W7 gate JSON → resolve `MERGED_REPO`.
3. Patch `config/300-final-blindset-B.yaml` (and 220 for the dev smoke) to point at `MERGED_REPO`.
4. Pytest pre-flight (W1–W7).
5. Dev smoke: `run_inference_devset.py --tid 220-responder-rgrpo-qwen7b-devset` → `exp/inference/devset/220-...json` → compute dev nDCG@20.
6. Blind-B inference: `run_inference_blindset.py --tid 300-final-blindset-B --eval_dataset blindset_B`.
7. Validate Blind-B output schema + package the zip.
8. Stage artifacts to Drive + emit a submission-ready summary.

In [ ]:
# 1) GPU check.
!nvidia-smi | head -20

In [ ]:
# 2) Clone fresh-model branch.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026
!git log -1 --pretty=format:'commit:  %h%nsubject: %s'

In [ ]:
# 2b) Drive + caches.
import os, shutil
from google.colab import drive
try: drive.mount('/content/drive')
except Exception as e:
    try: drive.flush_and_unmount()
    except Exception: pass
    drive.mount('/content/drive', force_remount=True)

DRIVE_BASE = '/content/drive/MyDrive/recsys2026-cache'
for d in [f'{DRIVE_BASE}/hf_datasets', f'{DRIVE_BASE}/experiments_cache',
          f'{DRIVE_BASE}/blindset_runs']:
    os.makedirs(d, exist_ok=True)

os.environ['HF_DATASETS_CACHE'] = f'{DRIVE_BASE}/hf_datasets'
%env HF_DATASETS_CACHE={DRIVE_BASE}/hf_datasets

EXPECTED_CACHE = '/content/recsys2026/music-crs-baselines/experiments/cache'
os.makedirs(os.path.dirname(EXPECTED_CACHE), exist_ok=True)
if os.path.exists(EXPECTED_CACHE) and not os.path.islink(EXPECTED_CACHE):
    shutil.rmtree(EXPECTED_CACHE)
if not os.path.islink(EXPECTED_CACHE):
    os.symlink(f'{DRIVE_BASE}/experiments_cache', EXPECTED_CACHE)

In [ ]:
# 3) HF auth — required for push_to_hub.
#
# Setup: Colab → 🔑 Secrets pane → add `HF_TOKEN` with WRITE scope.
# Get the token at https://huggingface.co/settings/tokens.
#
# Fail-fast: aborts immediately if the secret is missing or the token is
# invalid — better than failing 3 hours into training when push_to_hub fires.
import os, sys
from google.colab import userdata
from huggingface_hub import whoami

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception as e:
    raise SystemExit(
        f"\u274c HF_TOKEN secret not found in Colab ({e!r}).\n"
        f"   1) Open the \U0001f511 Secrets pane in the left sidebar.\n"
        f"   2) Add a secret named exactly `HF_TOKEN` (case-sensitive).\n"
        f"   3) Toggle 'Notebook access' ON for this notebook."
    )

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN

try:
    user = whoami(token=HF_TOKEN)
    HF_USERNAME = user["name"]
    os.environ["HF_USERNAME"] = HF_USERNAME
    print(f"\u2713 HF auth ok \u2014 logged in as {HF_USERNAME}")
except Exception as e:
    raise SystemExit(
        f"\u274c HF auth failed: {e!r}\n"
        f"   Token may lack WRITE scope. Regenerate at https://huggingface.co/settings/tokens"
    )

In [ ]:
# 4) Resolve W7 merged repo from gate_result.json.
import json
from pathlib import Path

def latest_gate(runs_dir: str):
    p = Path(runs_dir)
    if not p.exists(): return None
    candidates = list(p.rglob('gate_result.json'))
    if not candidates: return None
    latest = max(candidates, key=lambda x: x.stat().st_mtime)
    with latest.open() as f: return json.load(f), latest

W7_RESULT = latest_gate(f'{DRIVE_BASE}/grpo_final_runs')
W6_RESULT = latest_gate(f'{DRIVE_BASE}/grpo_runs')

if W7_RESULT:
    w7, _ = W7_RESULT
    W7_MERGED = w7['merged_hub_model']
    USING = 'W7 (train+dev retrain)'
    print(f'✓ W7 merged found: {W7_MERGED}')
    print(f'  format compliance: {w7.get("format_compliance_strict", 0):.1%}')
    if w7.get('format_compliance_strict', 0) < 0.95:
        print('⚠️  W7 format compliance < 95%. Consider falling back to W6.')
        FALLBACK = True
    else:
        FALLBACK = False
elif W6_RESULT:
    print('⚠️  No W7 gate found. Falling back to W6 merged repo for Blind-B.')
    W7_MERGED = W6_RESULT[0]['merged_hub_model']
    USING = 'W6 (W7 not yet run)'
    FALLBACK = True
else:
    raise SystemExit('❌ No W6 or W7 merged repo available. Run notebook 32 (and ideally 33) first.')

print(f'\nUSING for Blind-B: {USING} → {W7_MERGED}')

In [ ]:
# 5) Patch configs to point at the resolved merged repo.
#
# Both configs ship with a PLACEHOLDER lm_type. We rewrite the actual yaml
# files in-place (in the cloned repo) so the runners load the right model.
import re

def patch_lm_type(yaml_path: str, new_lm_type: str):
    with open(yaml_path, encoding='utf-8') as f: txt = f.read()
    new_txt = re.sub(r'^lm_type:\s*"[^"]*"', f'lm_type: "{new_lm_type}"', txt, flags=re.MULTILINE)
    if new_txt == txt:
        raise RuntimeError(f'failed to rewrite lm_type in {yaml_path} (regex did not match)')
    with open(yaml_path, 'w', encoding='utf-8') as f: f.write(new_txt)
    print(f'  ✓ patched {yaml_path} → lm_type: {new_lm_type}')

CONFIG_300 = '/content/recsys2026/music-crs-baselines/config/300-final-blindset-B.yaml'
CONFIG_220 = '/content/recsys2026/music-crs-baselines/config/220-responder-rgrpo-qwen7b-devset.yaml'
patch_lm_type(CONFIG_300, W7_MERGED)
patch_lm_type(CONFIG_220, W7_MERGED)

# Quick sanity print.
!grep '^lm_type:' /content/recsys2026/music-crs-baselines/config/300-final-blindset-B.yaml
!grep '^lm_type:' /content/recsys2026/music-crs-baselines/config/220-responder-rgrpo-qwen7b-devset.yaml

In [ ]:
# 6) Install deps + pytest pre-flight.
!pip install -q --upgrade transformers datasets 'pandas<3.0' tqdm omegaconf pyyaml
!pip install -q --upgrade 'trl>=0.12.0' 'peft>=0.13.0' 'torchao>=0.16.0' vllm

!cd /content/recsys2026 && python -m pytest \
    tests/test_reward_fns.py \
    tests/test_state_tracker.py \
    tests/test_cmqr.py \
    tests/test_pro_rank.py \
    tests/test_augment_envelope.py \
    tests/test_build_trl_datasets.py \
    tests/test_build_sdpo_dataset.py \
    tests/test_build_grpo_dataset.py \
    tests/test_build_train_plus_dev.py \
    tests/test_prediction_validator.py \
    -q

In [ ]:
# 7) Dev no-regression smoke (plan §6.6 hard rule: dev nDCG@20 must not regress > 0.005).
#
# Runs run_inference_devset.py with config 220 → exp/inference/devset/220-….json
# Then computes the leaderboard nDCG@20 via music-crs-evaluator/metrics/metrics_recsys.get_ndcg.
# The W3 retrieval-only baseline (frozen wRRF top-20) is in
# `exp/inference/devset/110-prorank-rerank-devset.json` — if that file exists in
# the clone, we compare. Else we just print the absolute number for review.
import os
DEV_OUT = '/content/recsys2026/music-crs-baselines/exp/inference/devset/220-responder-rgrpo-qwen7b-devset.json'

if Path(DEV_OUT).exists():
    print(f'reusing existing {DEV_OUT}')
else:
    !cd /content/recsys2026/music-crs-baselines && python run_inference_devset.py \
        --tid 220-responder-rgrpo-qwen7b-devset \
        --batch_size 16 --device cuda --attn_implementation sdpa

# Compute dev nDCG@20 against the leaderboard's gold.
import json
import sys
sys.path.insert(0, '/content/recsys2026/music-crs-evaluator')
from metrics.metrics_recsys import get_ndcg

# Load gold from the dev split's `music`-role rows.
from datasets import load_dataset
dev_raw = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='test')
gold_per_row = {}
for sess in dev_raw:
    sid = sess['session_id']
    for msg in sess['conversations']:
        if msg['role'] == 'music':
            gold_per_row[(sid, int(msg['turn_number']))] = str(msg['content'])

with open(DEV_OUT, encoding='utf-8') as f: dev_preds = json.load(f)
ndcg20s = []
missing_gold = 0
for row in dev_preds:
    key = (row['session_id'], int(row['turn_number']))
    g = gold_per_row.get(key)
    if g is None:
        missing_gold += 1
        continue
    pred_ids = row.get('predicted_track_ids') or []
    ndcg20s.append(get_ndcg([g], list(pred_ids), 20))

dev_ndcg20 = sum(ndcg20s) / max(len(ndcg20s), 1)
print(f'\nDEV nDCG@20 (post-W7): {dev_ndcg20:.4f}  ({len(ndcg20s):,} scored, {missing_gold} missing gold)')

# Compare to retrieval-only baseline if present.
BASELINE_FILE = '/content/recsys2026/music-crs-baselines/exp/inference/devset/110-prorank-rerank-devset.json'
if Path(BASELINE_FILE).exists():
    with open(BASELINE_FILE, encoding='utf-8') as f: base_preds = json.load(f)
    base_ndcg20s = []
    for row in base_preds:
        key = (row['session_id'], int(row['turn_number']))
        g = gold_per_row.get(key)
        if g is None: continue
        base_ndcg20s.append(get_ndcg([g], list(row.get('predicted_track_ids') or []), 20))
    base_ndcg20 = sum(base_ndcg20s) / max(len(base_ndcg20s), 1)
    delta = dev_ndcg20 - base_ndcg20
    print(f'BASELINE nDCG@20 (W3 ProRank): {base_ndcg20:.4f}')
    print(f'Δ vs baseline: {delta:+.4f}  (gate: must NOT regress > 0.005, so Δ ≥ -0.005)')
    if delta < -0.005:
        print(f'\n❌ W7 REGRESSED dev nDCG@20 by {-delta:.4f} — DO NOT submit to Blind-B.')
        print(f'   Falling back to W6 / W4 merged repo recommended.')
        DEV_GATE_PASSED = False
    else:
        print('✓ dev nDCG@20 no-regression gate PASSED.')
        DEV_GATE_PASSED = True
else:
    print(f'⚠️  no W3 baseline file at {BASELINE_FILE}; skipping no-regression check.')
    DEV_GATE_PASSED = True

In [ ]:
# 8) Blind-B inference.
#
# Plan §10 line 611: python music-crs-baselines/run_inference_blindset.py --tid 300-final-blindset-B
BLIND_B_OUT = '/content/recsys2026/music-crs-baselines/exp/inference/blindset_B/300-final-blindset-B.json'

if not DEV_GATE_PASSED:
    raise SystemExit('❌ Dev gate failed; refusing to spend a Blind-B submission slot.')

if Path(BLIND_B_OUT).exists():
    print(f'reusing existing {BLIND_B_OUT}')
else:
    !cd /content/recsys2026/music-crs-baselines && python run_inference_blindset.py \
        --tid 300-final-blindset-B \
        --eval_dataset blindset_B \
        --batch_size 16 --device cuda --attn_implementation sdpa

assert Path(BLIND_B_OUT).exists(), f'blindset_B run did not produce {BLIND_B_OUT}'
import json
with open(BLIND_B_OUT, encoding='utf-8') as f: blind_b = json.load(f)
print(f'\nBlind-B rows produced: {len(blind_b):,}')
print(f'sample row keys: {sorted(blind_b[0].keys()) if blind_b else "EMPTY!"}')

In [ ]:
# 9) Validate schema + package the CodaBench zip.
#
# Per project_codabench_submission.md: the uploaded zip must contain a single
# top-level entry `prediction.json`. validate_prediction.py.package_zip handles
# this via zipfile.writestr('prediction.json', payload).
from datetime import date
SUBMISSION_DIR = '/content/recsys2026/data/submissions'
os.makedirs(SUBMISSION_DIR, exist_ok=True)

ZIP_PATH = f'{SUBMISSION_DIR}/blindset_B_{date.today().isoformat()}_300.zip'

# 1. Validate schema (blindB).
!cd /content/recsys2026 && python scripts/validate_prediction.py \
    --input {BLIND_B_OUT} --split blindB --weekly-cap 3

# 2. Validate + package together.
import sys
sys.path.insert(0, '/content/recsys2026/scripts')
from validate_prediction import load_prediction, validate_schema, package_zip

predictions = load_prediction(BLIND_B_OUT)
errors = validate_schema(predictions, 'blindB')
if errors:
    print('\n❌ Schema validation FAILED:')
    for e in errors[:20]: print(f'  - {e}')
    raise SystemExit('Refusing to package — fix the inference output and rerun.')

print(f'\n✓ schema OK ({len(predictions):,} rows)')
out_zip = package_zip(BLIND_B_OUT, ZIP_PATH)
print(f'✓ packaged → {out_zip}')

# Stage to Drive for download.
import shutil
drive_zip = f'{DRIVE_BASE}/blindset_runs/blindset_B_{date.today().isoformat()}_300.zip'
shutil.copy(ZIP_PATH, drive_zip)
print(f'✓ Drive copy → {drive_zip}')

# Verify the zip layout.
import zipfile
with zipfile.ZipFile(ZIP_PATH) as zf:
    members = zf.namelist()
print(f'\nzip members: {members}')
assert members == ['prediction.json'], f'❌ wrong layout — expected [prediction.json], got {members}'
print('✓ zip layout OK (prediction.json at root, no parent dir)')

In [ ]:
# 10) Final summary.
import os
print('=' * 60)
print('W7 BLIND-B SUBMISSION READY')
print('=' * 60)
print(f'Source model:     {W7_MERGED}')
print(f'Source stage:     {USING}')
print(f'Dev gate:         {"PASSED" if DEV_GATE_PASSED else "FAILED"}')
if "dev_ndcg20" in dir():
    print(f'Dev nDCG@20:      {dev_ndcg20:.4f}')
print(f'Rows submitted:   {len(predictions):,}')
print(f'CodaBench zip:    {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1024:.1f} KB)')
print(f'Drive copy:       {drive_zip}')
print()
print('Next steps:')
print('  1. Download the zip from Drive.')
print('  2. Upload to CodaBench (≤1/week per plan §2.6).')
print('  3. Append a row to documents/submissions_log.md once accepted.')

## After this notebook

**On a successful CodaBench score:**
- Append a row to `documents/submissions_log.md` (date, source-stage, dev nDCG@20, leaderboard score). The next blind submission's budget check (`validate_prediction --check-budget`) reads this log.
- If score ≥ exp-021 anchor → W7 is the W8 starting point.
- If score < exp-021 → consider reverting to W6 merged for the next slot.

**On a Blind-B rejection (schema, count, format):**
- Do NOT count it against the 1/week cap (CodaBench typically returns those slots, but verify in the portal).
- Re-check `validate_prediction.py --split blindB`'s output for the actual mismatch; common culprits are predicted_track_ids length 0 (catalog filter dropped everything → backfill failure) or empty session/turn fields.

**Plan §10 final integration test reminder:** the W7 end-to-end smoke runs against a 100-session subset of dev. To do that, edit cell 7's `run_inference_devset.py` invocation to point at a small config variant (e.g. `220-...-devset-100`) once we have one. Not blocking for the first Blind-B submission.